# 08 — Cập nhật model: cải thiện vật nhận sai (mode `improve`)

[Plan](../../docs/project/feedback-finetune-plan-2026-11-01.md) · [kết quả verify](../../docs/project/feedback-finetune-verify-report-2026-11-01.md). Thay cho relearn task 6 (alias), đã dừng ở cổng với cAcc-M2 +1,02 điểm.

Chạy lại **task 5** của config gốc, khởi đầu từ model task 5 của round 1, trên dữ liệu có nhãn cho mọi vật: ảnh Val-A model đếm sai (phản hồi, ×3), ảnh ghép của các SKU trong cặp nhầm, và 30% replay phủ đủ 200 SKU. Chỉ ô phân loại của các SKU trong cặp nhầm được train.

| # | Nguyên nhân lỗi task 6 | Chỗ sửa |
|---|---|---|
| F1 | Tạo ô mới, không sửa ô cũ | `--train_rows`, chạy lại task 5 từ checkpoint task 5 |
| F2 | Ô SKU anh em bị khoá | `--train_rows` gồm cả hai phía của cặp |
| F3 | Học lại từ đầu | `--init_new_prompts 0`, giữ trọng số cũ |
| F4 | Nhãn giả mang lỗi của model cũ | `--pseudo none` + nhãn thật |
| F5 | Phần dùng chung bị kéo lệch | `--new_params class_embed,prompts.private`; replay |
| F6 | Learning rate cao, không giảm | `--lr 2e-5 --lr_old 5e-6` + `--lr_drop` |

**Cổng** (so với mốc trên Val-B; fixed / broken = ảnh sai→đúng / đúng→sai):

| Điều kiện | Pilot | E1 | Vòng 2 (so với E1) |
|---|---|---|---|
| cAcc-M2 Val-B | > mốc | ≥ mốc + 0,0165 | ≥ E1 |
| Sign test fixed > broken | p < 0,05 | p < 0,05 | — |
| Nhầm trong các cặp (hai chiều) | giảm ≥ 30% | giảm ≥ 30% | không tăng |
| AP50 SKU khác | giảm ≤ 0,01 | giảm ≤ 0,01 | giảm ≤ 0,005 |
| SKU (≥ 30 vật) tụt > 0,02 | ≤ 3 | 0 | 0 |

Một cell báo `GateStop` nghĩa là cổng đã dừng ở đó. Bị ngắt giữa chừng thì chạy lại file này: stage đã xong được bỏ qua. Test2019 chỉ chấm một lần, khi người dùng đổi `CONFIRM_TEST = True`.

In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import update_model
from autocheckout.io import load_json
from autocheckout.taskcfg import TaskConfig
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'configs/training/update_model_improve.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = update_model.prepare(SETTINGS)
NAMES = TaskConfig.load(PLAN['task_config']).label_names()
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Seed (round 1, task 5):', PLAN['seed_task5'], PLAN['seed'])
print('Relearn release:', PLAN['relearn_release'], '| selection:', len(PLAN['selection']), 'SKU')
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'num_workers', 'tf32',
                                         'pilot_pairs', 'e1_min_count', 'feedback_repeat', 'replay_share',
                                         'min_per_class')}, indent=1))

In [ ]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

## 1. Chia val (I2) và mốc round 1 (I10)

Val-A làm luồng phản hồi, Val-B chỉ để chấm; cả nhóm cảnh nằm một bên, cân theo mức easy/medium/hard. Mốc = model task 5 của round 1 chấm lại trên Val-B. Từ Val-A: ảnh model đếm sai (phản hồi) và các cặp SKU hay nhầm, ra danh sách ô cần train của pilot và E1.

In [ ]:
SPLIT = update_model.stage_split(PLAN)
print('Val-A:', SPLIT['val_a']['images'], SPLIT['val_a']['levels'], '| Val-B:', SPLIT['val_b']['images'], SPLIT['val_b']['levels'])
BASE = update_model.stage_baseline(PLAN)
print(f"round 1 on Val-B: cAcc-M2 {BASE['cAcc_m2']:.4f} | Val-A images counted wrong: {BASE['val_a_wrong_images']}")
print('rows: pilot', len(BASE['rows']['pilot']), '| e1', len(BASE['rows']['e1']))
print(f"{'pair':48s} {'confusions':>10s}")
for a, b, n in BASE['pairs_top50'][:15]:
    print(f"{NAMES[a][:22]:22s} / {NAMES[b][:22]:22s} {n:10d}")

In [ ]:
def show(stage):
    """Scores of a finished improve run, also after a stop; nothing if it did not finish (see the traceback)."""
    path = Path(PLAN['root']) / 'stages' / f'{stage}.json'
    if not path.is_file():
        return
    r = load_json(path)
    c, d = r['comparison'], r['data']
    print(f"{stage}: gate {r['gate']['decision'].upper()} - {r['gate']['reason']}")
    print(f"data: {d['images']} images ({d['feedback_images']} feedback x{PLAN['feedback_repeat']}, "
          f"{d['synthetic_images']} synthetic, {d['replay_images']} replay), {len(r['rows'])} rows")
    print(f"cAcc-M2 {c['cAcc_base']:.4f} -> {c['cAcc_new']:.4f} | fixed {c['fixed']} broken {c['broken']} "
          f"p {c['sign_test_p']:.3g} | AP50 other {r['AP50_other']['base']:.4f} -> {r['AP50_other']['new']:.4f}")
    print('SKUs that fell:', [(NAMES[s['label']], round(s['base'], 3), round(s['new'], 3)) for s in c['falls']])
    for p in c['top_pairs'][:10]:
        print(f"  {NAMES[p['pair'][0]][:22]:22s} / {NAMES[p['pair'][1]][:22]:22s} {p['base']:5d} -> {p['new']:5d}")

## 2. Pilot: 10 cặp nhầm nặng nhất (khoảng 2 giờ trên 4090)

4 epoch, `--lr_drop 3`. `STOP` → dừng, báo người dùng; không tự chạy E2.

In [ ]:
try:
    update_model.stage_train(PLAN, 'pilot')
finally:
    show('pilot')

## 3. E1: mọi cặp ≥ 10 lần nhầm + 50 SKU của relearn (khoảng 9 giờ)

Chỉ chạy sau `GO` của pilot. 6 epoch, `--lr_drop 4`. Nếu `STOP`: hỏi người dùng có chạy E2 (mở cả 200 ô) không, rồi đổi `RUN_E2 = True`.

In [ ]:
try:
    update_model.stage_train(PLAN, 'e1')
finally:
    show('e1')

In [ ]:
RUN_E2 = False
if RUN_E2:
    try:
        update_model.stage_train(PLAN, 'e2')
    finally:
        show('e2')

## 4. Vòng phản hồi thứ 2 (khoảng 1 giờ)

Ảnh Val-A mà model E1 vẫn đếm sai thành phản hồi mới; 2 epoch từ model E1, so với E1.

In [ ]:
try:
    update_model.stage_round2(PLAN)
finally:
    show('round2')

## 5. Test2019 (một lần)

Model mới nhất đã `GO` (vòng 2, rồi E1, rồi E2), dùng policy chọn trên Val-B. Đọc các cổng ở trên, rồi tự đổi `CONFIRM_TEST = True` và chạy lại notebook.

In [ ]:
CONFIRM_TEST = False
FINAL = update_model.stage_final_test(PLAN, confirm=CONFIRM_TEST or PLAN['dry_run'])
print('model:', FINAL['model'], '| Val-B cAcc-M2:', FINAL['val_b_cAcc_m2'])
print('test AP50:', FINAL['test2019']['AP50'])
for policy, stages in FINAL['test2019']['counting'].items():
    print(policy, json.dumps(stages))